In [ ]:
# import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import liana as li

from liana.method import cellphonedb, connectome

from pathlib import Path
#import session_info

In [ ]:
adata = sc.read("./VisiumHD_NOTwithref_annoated.h5ad")

In [ ]:
adata.uns['celltype_final_colors'] = ['#1f77b4', '#abddde', '#00ffff', '#fc6fcf', '#cc66ff', '#f297a7', '#289e92', '#ff2600', '#17becf', '#d7ef9b', 
                                     '#ffff0a', '#e6994d', '#ff9896', '#5084c2', '#80ff08']

In [ ]:
sc.pl.spatial(adata, color=['celltype_final'], size=1.5, img_key=None)

In [ ]:
sc.set_figure_params(figsize=(5, 8))
plt.scatter(adata.obs['x'], adata.obs['y'], s = 0.01)

In [ ]:
adata_sub = adata[(adata.obs['x'] >= 16000) * (adata.obs['x'] <= 24000) * (adata.obs['y'] >= 26000) * (adata.obs['y'] <= 29000), :].copy()
adata_sub

In [ ]:
sc.set_figure_params(figsize=(5, 5))
plt.scatter(adata_sub.obs['x'], adata_sub.obs['y'], s = 0.01)

In [ ]:
adata_sub

In [ ]:
adata_sub.X = adata_sub.layers['counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pl.umap(adata_sub, color='celltype_final')

### Run MISTy

In [ ]:
#!pip install decoupler

In [ ]:
import decoupler as dc
# obtain genesets
progeny = dc.op.progeny(organism='human', top=500)

In [ ]:
from liana.method import MistyData, genericMistyData, lrMistyData

In [ ]:
# use multivariate linear model to estimate activity
dc.mt.mlm(
    adata_sub,
    net=progeny,
    verbose=True,
    raw=False
)

In [ ]:
acts_progeny = li.ut.obsm_to_adata(adata_sub, 'score_mlm')

In [ ]:
adata_sub.obsm['score_mlm']

In [ ]:
acts_progeny

In [ ]:
sc.pl.spatial(acts_progeny, color=adata_sub.obsm['score_mlm'].columns, cmap='RdBu_r', size=1.3, img_key=None)

In [ ]:
acts_progeny = sc.read_h5ad("./VisiumHD_acts_progeny.h5ad")

In [ ]:
sc.pl.violin(acts_progeny, keys=adata_sub.obsm['score_mlm'].columns, groupby="celltype_final", rotation=90)

In [ ]:
ax_list = sc.pl.violin(acts_progeny, keys=adata_sub.obsm['score_mlm'].columns, groupby="celltype_final", rotation=90, show=False)
if isinstance(ax_list, list):
    for ax in ax_list:
        ax.set_ylim(-4, 6) # Set desired y-limits for each axis
else:
    # If only one plot was generated, it might return a single Axes object
    ax_list.set_ylim(-4, 6)
plt.show()

In [ ]:
adata_sub.write_h5ad("./VisiumHD_NOTwithref_annoated_ccc.h5ad")

In [ ]:
adata_sub = sc.read_h5ad("./VisiumHD_NOTwithref_annoated_ccc.h5ad")
adata_sub

In [ ]:
acts_progeny.write_h5ad("./VisiumHD_acts_progeny.h5ad")